In [ ]:
import pandas as pd
import IPython.display as display
import yfinance as yf
import re
import numpy as np

np.random.seed(927)

# Portfolio Data Cleaning and Analysis
This notebook loads, cleans, and analyzes portfolio data from various sources, including equities, ETFs, mutual funds, and bonds. It also demonstrates portfolio analytics such as volatility, returns, Sharpe ratio, and efficient frontier simulation.

# Load initial data

---
## 1. Import and Clean Data
Load raw data from CSVs, clean column names, and standardize formats for further analysis.

In [ ]:
def to_snake_case(s: str) -> str:
    s = s.strip()
    s = re.sub(r"(?<=[a-z0-9])([A-Z])", r" \1", s)
    s = re.sub(r"[^\w]+", "_", s)  # replace spaces & symbols with _
    s = re.sub(r"_+", "_", s)  # collapse multiple _
    return s.lower().strip("_")


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # cleaning the column names and converting to snake case
    df.columns = [to_snake_case(c) for c in df.columns]

    # clean string values
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].str.strip()

    return df


# Load the equity master csv for mapping isin to symbol
nse_eq = clean_df(pd.read_csv("../data/raw/nse_eq_master.csv"))

# Making column names more consistent/usable
nse_eq.rename(
    columns={"isin_number": "isin", "date_of_listing": "list_date"}, inplace=True
)

# Load the etf master csv for mapping isin to symbol
nse_etf = clean_df(pd.read_csv("../data/raw/nse_etf_master.csv"))
nse_etf.rename(
    columns={"isinnumber": "isin", "dateof_listing": "list_date"}, inplace=True
)

# Load broker csv
broker = clean_df(pd.read_csv("../data/raw/current_portfolio.csv"))

display.display(broker)
display.display(nse_eq.head())
display.display(nse_etf.head())

Some rows in the broker DF be indicating the type of security and are not useful for analysis. We will remove these. We will also use simple labels for security type instead of the ones provided. 

---
### 2. Clean Broker Data
Remove non-informative rows and standardize security type labels for easier downstream processing.

In [ ]:
# remove rows with NaN in ISIN column from the broker df
broker = broker[broker["isin"].notna()]
broker["security_type"] = broker["security_type"].replace(
    {"EQUITY STOCK": "EQ", "MUTUAL FUND": "MF", "BONDS": "BOND"}
)
broker

Some columns in the NSE DFs also seem inconsistently formatted, let's analyze this further.

---
### 3. Explore and Clean NSE Data
Check for inconsistencies and convert date columns to datetime for analysis.

In [ ]:
nse_eq.info()
display.display(nse_eq.describe(include="all"))

In [ ]:
nse_etf.info()
display.display(nse_etf.describe(include="all"))

We need to convert the list_date which is currently stored as strings to datetime to make it easier to analyze. 

In [ ]:
def convert_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    s = df["list_date"]

    dt = pd.to_datetime(s, format="%d-%b-%y", errors="coerce")
    dt = dt.fillna(pd.to_datetime(s, format="%d-%b-%Y", errors="coerce"))

    df["list_date"] = dt

    assert df["list_date"].notna().all()

    return df


nse_eq, nse_etf = convert_dates(nse_eq), convert_dates(nse_etf)

display.display(nse_eq.head())
display.display(nse_etf.head())

---
### 4. Split Portfolio by Security Type
Separate equities, ETFs, mutual funds, and bonds for targeted analysis.

In [ ]:
eq = broker[broker.security_type == "EQ"].copy()
etf = broker[broker.security_type == "ETF"].copy()
mf = broker[broker.security_type == "MF"].copy()
bond = broker[broker.security_type == "BOND"].copy()
display.display(eq)
display.display(etf)
display.display(mf)
display.display(bond)

In [ ]:
eq = eq.merge(nse_eq[["isin", "symbol", "list_date"]], on="isin", how="left")
eq

---
### 5. Map ISIN to Symbol and Merge Data
Merge broker data with NSE master data to get symbols and listing dates for equities and ETFs.

In [ ]:
etf = etf.merge(nse_etf[["isin", "symbol", "list_date"]], on="isin", how="left")
etf

In [ ]:
eq_etf = pd.concat([eq, etf], ignore_index=True)
eq_etf["symbol"] = eq_etf["symbol"] + ".NS"
eq_etf

---
### 6. Fetch Latest Prices and Build Canonical Table
Download latest prices using yfinance, compare with broker prices, and build a canonical table for further analysis.

In [ ]:
yf_1d_data = yf.download(
    eq_etf["symbol"].tolist(), period="1d", progress=False, threads=True
)

latest_close = yf_1d_data["Close"].iloc[-1]

eq_etf["fetched_price"] = eq_etf["symbol"].map(latest_close)

eq_etf

In [ ]:
eq_etf["broker_minus_fetched"] = eq_etf["current_price"] - eq_etf["fetched_price"]
eq_etf

In [ ]:
eq_etf.columns

In [ ]:
eq_etf_canon = eq_etf.loc[
    :,
    [
        "isin",
        "symbol",
        "security_type",
        "sector",
        "quantity",
        "fetched_price",
    ],
].assign(current_value=lambda d: d["quantity"] * d["fetched_price"])

eq_etf_canon

We now have a possible canonical table that we can use as a standard, even when we have to work with differently formatted broker data (only for equities and etfs). Once everything looks like this, we can use the same steps moving forward.

## Brief Visualization before proceeding

---
## 2. Portfolio Visualization
Visualize the current portfolio allocation and download historical price data for further analysis.

In [ ]:
import matplotlib.pyplot as plt

# Build weights with symbol as index
weights = (
    eq_etf_canon.set_index("symbol")["current_value"]
    .div(eq_etf_canon["current_value"].sum())
    .sort_values(ascending=True)
)

plt.figure(figsize=(8, 6))
ax = weights.plot(kind="barh")

# Add percentage labels on bars
for i, v in enumerate(weights):
    ax.text(v, i, f" {v:.1%}", va="center")

plt.xlabel("Portfolio Weight")
plt.title("Current Portfolio Allocation")
plt.tight_layout()
plt.show()

In [ ]:
history = yf.download(
    eq_etf_canon["symbol"].to_list(),
    period="10y",
    interval="1d",
    auto_adjust=True,
    progress=False,
)

history

In [ ]:
history.info()
print(history.shape)

---
### 7. Add SGB Data
Process Sovereign Gold Bond (SGB) data and append to the canonical table for a complete portfolio view.

In [ ]:
sgb = pd.read_csv("../data/raw/SGBMAY28.csv")

# parse Date
sgb["Date"] = pd.to_datetime(sgb["Date"], format="%d-%B-%Y", errors="coerce")

# keep needed columns
sgb = sgb[["Date", "Close Price"]].copy()

# rename price column
sgb = sgb.rename(columns={"Close Price": "SGBMAY28"})

# set datetime index (only once)
sgb = sgb.set_index("Date").sort_index()

# ensure numeric dtype
sgb["SGBMAY28"] = pd.to_numeric(sgb["SGBMAY28"], errors="coerce")

sgb.tail()

In [ ]:
close_history = history.Close.copy()
close_history = close_history.join(sgb, how="left")

In [ ]:
close_history.info()
close_history.describe()

In [ ]:
# Ffill to deal with missing values after inception, these are generally small windows
close_history = close_history.ffill()
close_history.info()

# TODO
Move this to its correct place (This needs to be placed earlier)

In [ ]:
# --- take the single SGB row ---
sgb_row = bond.copy()

# --- standardize fields to match canon ---
sgb_row = (
    sgb_row.rename(
        columns={
            "script_name": "script_name",
            "isin": "isin",
            "security_type": "security_type",
            "sector": "sector",
            "quantity": "quantity",
        }
    )
    .assign(
        symbol="SGBMAY28",  # must match price column in close_history
        sector="Gold",  # override generic "Others"
        fetched_price=close_history["SGBMAY28"].iloc[-1],
        current_value=lambda d: d["quantity"] * d["fetched_price"],
    )
    .loc[
        :,
        [
            "isin",
            "symbol",
            "security_type",
            "sector",
            "quantity",
            "fetched_price",
            "current_value",
        ],
    ]
)

# --- append into existing canon ---
canon = pd.concat([eq_etf_canon, sgb_row], ignore_index=True)

canon.tail()

In [ ]:
w = canon.set_index("symbol")

In [ ]:
daily_returns = close_history / close_history.shift(1) - 1
daily_returns

---
## 3. Return Calculation and Outlier Cleaning
Calculate daily returns and clean outliers to ensure robust risk and return analysis.

In [ ]:
daily_returns.describe()

### Outliers
There seem to be some outliers in our fetched prices because HNGSNGBEES and MON100 seem to have more than 800% returns on some day. We should therefore check for outliers in our returns and take measures to have more consistent data.

In [ ]:
def clean_outliers(stock: pd.Series) -> pd.Series:
    median = stock.median()
    mad = (stock - median).abs().median()
    robust_sigma = 1.4826 * mad
    z = (stock - median) / robust_sigma

    extreme_mask = z.abs() >= 8

    if not extreme_mask.any():
        return stock

    # WINSORIZE: Cap extreme values instead of removing data
    cleaned = stock.copy()
    threshold = 8 * robust_sigma
    lower_bound = median - threshold
    upper_bound = median + threshold

    cleaned = cleaned.clip(lower=lower_bound, upper=upper_bound)

    return cleaned


cleaned_returns = daily_returns.apply(clean_outliers)
cleaned_returns

In [ ]:
cleaned_returns.describe()

In [ ]:
covariance_matrix = cleaned_returns.cov() * 252
covariance_matrix

---
## 4. Covariance, Correlation, and Heatmap
Compute covariance and correlation matrices for asset returns, and visualize correlations using a heatmap.

In [ ]:
correlation_matrix = cleaned_returns.corr()
correlation_matrix

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 8))

sns.heatmap(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1, square=True)

plt.title("Correlation heatmap of Asset Returns")
plt.show()

---
## 5. Portfolio Volatility, Return, and Sharpe Ratio
Calculate annualized portfolio volatility, expected return, and Sharpe ratio using the cleaned data.

In [ ]:
w = canon.set_index("symbol")["current_value"]
w = w / w.sum()

w = w.reindex(covariance_matrix.columns)

assert np.isclose(w.sum(), 1), "Weights do not sum to 1"
assert (w.index == covariance_matrix.columns).all(), "Weights not aligned to covariance"

w

In [ ]:
w_vec = w.values
Sigma = covariance_matrix.values

portfolio_variance = w_vec.T @ Sigma @ w_vec

portfolio_vol_annual = np.sqrt(portfolio_variance)


print(
    f"Portfolio Variance: {portfolio_variance}\nAnnual Portfolio Volatility:{portfolio_vol_annual}"
)

In [ ]:
mu = cleaned_returns.mean() * 252

mu = mu.reindex(w.index)

portfolio_ret_annual = w.values @ mu.values


print("Annual Return =", portfolio_ret_annual)

$$Sharpe = (\mu _p - r_f)/\sigma _p$$

In [ ]:
risk_free = 0.067

sharpe = (portfolio_ret_annual - risk_free) / portfolio_vol_annual
print(f"Sharpe ratio = {sharpe}")

---
## 6. Monte Carlo Simulation for Efficient Frontier
Simulate random portfolios to visualize the efficient frontier and compare with the current portfolio.

In [ ]:
n_assets = len(w)
n_portfolios = 1_000_000
mu_vec = mu.values
gold_weight = 0.481196

weights_list = []
results = pd.DataFrame(np.zeros((n_portfolios, 3)), columns=["ret", "vol", "sharpe"])

for i in range(n_portfolios):
    trial_weights = np.random.dirichlet(
        np.ones(n_assets - 1)
    )  # Using dirichlet so that portfolios
    # with extreme concentrations are also considered

    trial_weights *= 1 - gold_weight
    # print(trial_weights.sum())

    trial_weights = np.append(trial_weights, [gold_weight])

    trial_vol = np.sqrt(trial_weights.T @ Sigma @ trial_weights)

    trial_ret = trial_weights @ mu_vec

    trial_sharpe = (trial_ret - risk_free) / trial_vol

    weights_list.append(trial_weights)
    results.iloc[i] = [trial_ret, trial_vol, trial_sharpe]


results.head()

In [ ]:
max_sharpe_idx = results["sharpe"].idxmax()
min_vol_idx = results["vol"].idxmin()

max_sharpe_point = results.loc[max_sharpe_idx]
min_vol_point = results.loc[min_vol_idx]

print(f"Max Sharpe point: {max_sharpe_point}")
print(f"min_vol_point: {min_vol_point}")

In [ ]:
results["sharpe"].describe()

In [ ]:
current_sharpe = sharpe  # your earlier value

percentile = (results["sharpe"] < current_sharpe).mean()
percentile

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull

plt.figure(figsize=(10, 6))

# --- Monte Carlo cloud ---
sc = plt.scatter(results["vol"], results["ret"], c=results["sharpe"], s=5)

plt.colorbar(sc, label="Sharpe Ratio")

points = results[["vol", "ret"]].values

hull = ConvexHull(points)

hull_points = points[hull.vertices]

# Sort hull points by volatility
hull_points = hull_points[np.argsort(hull_points[:, 0])]

# Keep only the upper frontier (monotonic increasing return)
frontier = []
max_ret = -np.inf

for vol, ret in hull_points:
    if ret > max_ret:
        frontier.append((vol, ret))
        max_ret = ret

frontier = np.array(frontier)

plt.plot(frontier[:, 0], frontier[:, 1], linewidth=3, label="Efficient Frontier")

# --- Key portfolios ---

# Current portfolio
plt.scatter(
    portfolio_vol_annual,
    portfolio_ret_annual,
    marker="o",
    s=120,
    label="Current Portfolio",
)

# Minimum variance
plt.scatter(
    min_vol_point["vol"], min_vol_point["ret"], marker="X", s=180, label="Min Variance"
)

# Maximum Sharpe
plt.scatter(
    max_sharpe_point["vol"],
    max_sharpe_point["ret"],
    marker="*",
    s=250,
    label="Max Sharpe",
)

# --- Labels & legend ---
plt.xlabel("Annual Volatility")
plt.ylabel("Annual Return")
plt.title("Monte Carlo Efficient Frontier with Key Portfolios")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
(0.175 + 0.15) / 2

In [ ]:
weights_max_sharpe = pd.Series(weights_list[max_sharpe_idx], index=mu.index)
weights_min_var = pd.Series(weights_list[min_vol_idx], index=mu.index)

In [ ]:
weights_max_sharpe

In [ ]:
weights_min_var

In [ ]:
pd.Series(w_vec, index=mu.index)

In [ ]:
weights_max_sharpe - w_vec

In [ ]:
weights_min_var - w_vec

---
## 7. Portfolio Optimization with CVXPY
Use convex optimization to find the optimal portfolio weights under constraints and compare with simulation results.

In [ ]:
import cvxpy as cp

num_securities = len(w_vec)

w_to_optimize = cp.Variable(num_securities)
r_min = 0.075

obj = cp.Minimize(w_to_optimize.T @ covariance_matrix @ w_to_optimize)
const = [
    cp.sum(w_to_optimize) == 1,
    w_to_optimize >= 0,
    w_to_optimize[-1] >= 0.4,
    # w_to_optimize[mu.index.get_loc('DRREDDY.NS')] >= 0.08,
    w_to_optimize[mu.index.get_loc("SILVERBEES.NS")] == 0,
    w_to_optimize.T @ mu.values - r_min >= 0.09,
]

prob = cp.Problem(obj, const)
optimized = prob.solve()

risk_opt = np.sqrt(optimized)
w_opt = w_to_optimize.value

print("optimal_risk:", risk_opt)
print("optimal weights:", np.round(w_opt, 4))

In [ ]:
optimized_series = pd.Series(np.round(w_opt, 4), index=mu.index)
optimized_series

In [ ]:
optimized_series.sum()

In [ ]:
optimized_series.values @ mu.values

---
## 8. Visualize Optimized Portfolio
Plot the optimized portfolio weights for a clear view of asset allocation.

In [ ]:
import matplotlib.ticker as mtick

# Plotting the series as a vertical bar graph
ax = optimized_series.plot(
    kind="bar", figsize=(10, 6), color="purple", edgecolor="black"
)

# Formatting labels and title
ax.set_title("Optimized Portfolio Weights", fontsize=14, fontweight="bold")
ax.set_xlabel("Assets", fontsize=12)
ax.set_ylabel("Weight", fontsize=12)

# Assuming your weights are decimals (e.g., 0.15 for 15%), format y-axis as percentage
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
mu

In [ ]:
optimized_series.to_dict()

In [ ]:
import itertools

[i for i in filter(lambda x: x % 5, itertools.islice(itertools.count(5), 10))]